# 1.5 Comparing measured and calculated pressures and fluid compositions from FI and MI data

We can now compare our results for Exercise 1 from VESIcal, VolFe, and DiadFit for melt and fluid inclusion data!

## 1.5.1 Import packages

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import DiadFit as pf

## 1.5.2 Import data

We're using the results from the previous notebooks in Exercise 1

In [ ]:
results_pvsat_vc = pd.read_csv("output/results_pvsat_vc.csv")
results_pvsat_vf = pd.read_csv("output/results_pvsat_vf.csv")
results_P_df = pd.read_csv("output/results_P_df.csv")

In [ ]:
# if you haven't run Exercise 1, you can grab the "answer key" files from here - remove the # at the start of the lines below and then run the cell
#results_pvsat_vc = pd.read_csv("answers/results_pvsat_vc.csv")
#results_pvsat_vf = pd.read_csv("answers/results_pvsat_vf.csv")
#results_P_df = pd.read_csv("answers/results_P_df.csv")

And the FI data from DeVitre & Wieser (2024)

In [ ]:
df_FI = pd.read_excel("input/Kilauea_MI_FI_MG_data.xlsx", sheet_name='FI_Data') # creates a DataFrame of just data in the FI_Data sheet

## 1.5.3 Calculate SO<sub>2</sub>/(SO<sub>2</sub>+CO<sub>2</sub>) for FI data

We can also compare the measured SO<sub>2</sub>/(SO<sub>2</sub>+CO<sub>2</sub>) ratio in the FI using Raman spectroscopy to the calculated ratio for MI using VolFe.

First we calculate the SO<sub>2</sub> proportion in the FI from the relative SO<sub>2</sub>-CO<sub>2</sub> peak areas using DiadFit:

In [ ]:
df_FI['SO2_Burke_table_pref']=pf.calculate_SO2_CO2_mol_prop_wave_indep(SO2_wavelength_ind=4.03, CO2_diad1_wavelength_ind=0.8, CO2_diad2_wavelength_ind=1.23, wavelength_nm=532.046,T_C=37,
                               A_SO2=df_FI['SO2_Area'], A_CO2_Tot=(df_FI['ν1_Voigt_Area']+ df_FI['2ν2_Voigt_Area']))

## 1.5.4 Plotting

We can compare the pressures calculated from MI data using VESIcal and VolFe and FI data using DiadFit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4)) # single panel figure

# clean the datasets for plotting - DiadFit will be in orange
datasets = [
    (results_pvsat_vc['SaturationP_bars_VESIcal'].dropna(), 'MI VESIcal-IM', 'red'),
    (results_pvsat_vf['P_bar'].dropna(), 'MI VolFe', 'purple'),
    ((results_P_df['P_kbar'] * 1000).dropna(), 'FI DiadFit', 'orange')
]

# plot as cumulative probability
for data, label, color in datasets:
    x = np.sort(data)
    y = np.arange(1, len(x) + 1) / len(x)
    ax.plot(x, y, lw=2, label=label, color=color)

# label axes
ax.set_xlabel('P (bars)')
ax.set_ylabel('Cumulative probability')

ax.set_ylim(0, 1) # set axes range
ax.legend() # add legend
ax.grid(alpha=0.3) # add grid

And the fluid and melt compositions!

In [ ]:
fig, ((ax1, ax2, ax3),(ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(12,8)) # six panel figure with three columns and two rows

# vesical results in red
df = results_pvsat_vc
ax1.plot(df['H2O'], df['SaturationP_bars_VESIcal'], 'ok', mfc='red')
ax2.plot(df['CO2']*10000, df['SaturationP_bars_VESIcal'], 'ok', mfc='red', label ="MI VESIcal_IM")
ax3.plot(df['XCO2_fl_VESIcal']*100, df['SaturationP_bars_VESIcal'], 'dk', mfc='red')

# volfe results in purple - not that VolFe uses different column names to VESIcal in its results
df = results_pvsat_vf
ax1.plot(df['H2OT-eq_wtpc'], df['P_bar'], 'ok', mfc='purple')
ax2.plot(df['CO2T-eq_ppmw'], df['P_bar'], 'ok', mfc='purple', label = "MI VolFe")
ax3.plot(df['xgCO2_mf']*100, df['P_bar'], 'dk', mfc='purple')
ax4.plot(df['ST_ppmw'], df['P_bar'], 'ok', mfc='purple')
ax5.plot(df['Fe3+/FeT'], df['P_bar'], 'ok', mfc='purple')
ax6.plot(100*df['xgSO2_mf']/(df['xgCO2_mf']+df['xgSO2_mf']), df['P_bar'], 'dk', mfc='purple')

# FI data
plt.plot(df_FI['SO2_Burke_table_pref']*100, results_P_df['P_kbar']*1000, 'dk', mfc='orange', label='FI DiadFit')

#axes labels
ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax2.set_xlabel('CO$_2$ [m] (ppm)')
ax3.set_xlabel('CO$_2$ [f] (mol%)')
ax4.set_ylabel('P (bars)')
ax4.set_xlabel('S [m] (ppm)')
ax5.set_xlabel('Fe$^{3+}$/Fe$_T$ [m]')
ax6.set_xlabel('SO$_2$/(CO$_2$+SO$_2$) [f] (mol%)')

#legend
ax2.legend(frameon=False)
ax6.legend(frameon=False)

plt.tight_layout()

## 1.5.5 Summary

In Exercise 1 we calculated temperatures for MI data using <a href="1_1_MI_Temperature_Thermobar.ipynb.ipynb">Thermobar</a> and pressures and fluid compositions from MI data using <a href="1_1_MI_Temperature_Thermobar.ipynb">VESIcal</a> and <a href="1_3_MI_Pressure_FluidComp_VolFe.ipynb">VolFe</a> and FI data using <a href="1_4_FI_Pressure_DiadFit.ipynb">DiadFit</a>.

For information on how we'd cite such calculations head to <a href="3_Describing_and_Citing_Calculations.ipynb">3. Describing and citing calculations</a>, which also contains the full references for papers mentioned in this exercise.